![Logo 1](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech1.jpg)
<div class="alert alert-block alert-info">
<h1> Komputerowe wspomaganie tłumaczenia </h1>
<h2> 3. <i>Terminologia</i> [laboratoria]</h2> 
<h3>Rafał Jaworski (2021)</h3>
</div>

![Logo 2](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech2.jpg)

Na dzisiejszych zajęciach zajmiemy się bliżej słownikami używanymi do wspomagania tłumaczenia. Oczywiście na rynku dostępnych jest bardzo wiele słowników w formacie elektronicznym. Wiele z nich jest gotowych do użycia w SDL Trados, memoQ i innych narzędziach CAT. Zawierają one setki tysięcy lub miliony haseł i oferują natychmiastową pomoc tłumaczowi.

Problem jednak w tym, iż często nie zawierają odpowiedniej terminologii specjalistycznej - używanej przez klienta zamawiającego tłumaczenie. Terminy specjalistyczne są bardzo częste w tekstach tłumaczonych ze względu na następujące zjawiska:
- Teksty o tematyce ogólnej są tłumaczone dość rzadko (nikt nie tłumaczy pocztówek z pozdrowieniami z wakacji...)
- Te same słowa mogą mieć zarówno znaczenie ogólne, jak i bardzo specjalistyczne (np. "dziedziczenie" w kontekście prawnym lub informatycznym)
- Klient używa nazw lub słów wymyślonych przez siebie, np. na potrzeby marketingowe.

Nietrywialnymi zadaniami stają się: odnalezienie terminu specjalistycznego w tekście źródłowym oraz podanie prawidłowego tłumaczenia tego terminu na język docelowy

Brzmi prosto? Spróbujmy wykonać ręcznie tę drugą operację.

### Ćwiczenie 1: Podaj tłumaczenie terminu "prowadnice szaf metalowych" na język angielski. Opisz, z jakich narzędzi skorzystałaś/eś.

Odpowiedź:

metal cabinet slides, użyłem wyszukiwarki i swojej wiedzy

W dalszych ćwiczeniach skupimy się jednak na odszukaniu terminu specjalistycznego w tekście. W tym celu będą potrzebne dwie operacje:
1. Przygotowanie słownika specjalistycznego.
2. Detekcja terminologii przy użyciu przygotowanego słownika specjalistycznego.

Zajmijmy się najpierw krokiem nr 2 (gdyż jest prostszy). Rozważmy następujący tekst:

In [1]:
text = " For all Java programmers:"
text += " This section explains how to compile and run a Swing application from the command line."
text += " For information on compiling and running a Swing application using NetBeans IDE,"
text += " see Running Tutorial Examples in NetBeans IDE. The compilation instructions work for all Swing programs"
text += " — applets, as well as applications. Here are the steps you need to follow:"
text += " Install the latest release of the Java SE platform, if you haven't already done so."
text += " Create a program that uses Swing components. Compile the program. Run the program."

Załóżmy, że posiadamy następujący słownik:

In [2]:
dictionary = ['program', 'application', 'applet', 'compile']

### Ćwiczenie 2: Napisz program, który wypisze pozycje wszystkich wystąpień poszczególnych terminów specjalistycznych. Dla każdego terminu należy wypisać listę par (pozycja_startowa, pozycja końcowa).

In [15]:
def terminology_lookup(text, dictionary):
    results = {}
    
    for term in dictionary:
        occurrences = []
        start_search = 0
        while True:
            idx = text.find(term, start_search)
        
            if idx == -1:
                break
        
            occurrences.append((idx, idx + len(term)))
            start_search = idx + 1
            
        results[term] = occurrences
    
    return results

found_terms = terminology_lookup(text, dictionary)
for term, positions in found_terms.items():
    print(f"{term}: {positions}")

program: [(14, 21), (291, 298), (468, 475), (516, 523), (533, 540)]
application: [(80, 91), (164, 175), (322, 333)]
applet: [(302, 308)]
compile: [(56, 63)]


Zwykłe wyszukiwanie w tekście ma pewne wady. Na przykład, gdy szukaliśmy słowa "program", złapaliśmy przypadkiem słowo "programmer". Złapaliśmy także słowo "programs", co jest poprawne, ale niepoprawnie podaliśmy jego pozycję w tekście.

Żeby poradzić sobie z tymi problemami, musimy wykorzystać techniki przetwarzania języka naturalnego. Wypróbujmy pakiet spaCy:

`pip3 install spacy`

oraz

`python3 -m spacy download en_core_web_sm`

In [4]:
!pip install spacy
!python -m spacy download en_core_web_sm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cache

In [5]:

import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

for token in doc:
    print(token.lemma_)

 
for
all
Java
programmer
:
this
section
explain
how
to
compile
and
run
a
Swing
application
from
the
command
line
.
for
information
on
compile
and
run
a
swing
application
use
NetBeans
IDE
,
see
run
Tutorial
Examples
in
NetBeans
IDE
.
the
compilation
instruction
work
for
all
Swing
program
—
applet
,
as
well
as
application
.
here
be
the
step
you
need
to
follow
:
install
the
late
release
of
the
Java
SE
platform
,
if
you
have
not
already
do
so
.
create
a
program
that
use
Swing
component
.
compile
the
program
.
run
the
program
.


Sukces! Nastąpił podział tekstu na słowa (tokenizacja) oraz sprowadzenie do formy podstawowej każdego słowa (lematyzacja).

### Ćwiczenie 3: Zmodyfikuj program z ćwiczenia 2 tak, aby zwracał również odmienione słowa. Na przykład, dla słowa "program" powinien znaleźć również "programs", ustawiając pozycje w tekście odpowiednio dla słowa "programs". Wykorzystaj właściwość idx tokenu.

In [10]:
def terminology_lookup(text, dictionary):
    doc = nlp(text)
    
    results = {term: [] for term in dictionary}
    for token in doc:
        lemma = token.lemma_.lower()
        
        if lemma in dictionary:
            start = token.idx
            end = start + len(token.text)
            
            results[lemma].append((start, end))
            
    return results

found_terms = terminology_lookup(text, dictionary)

for term, positions in found_terms.items():
    print(f"\nTerm: {term}")
    for pos in positions:
        original_word = text[pos[0]:pos[1]]
        print(f"{original_word=}, {pos=}")


Term: program
original_word='programs', pos=(291, 299)
original_word='program', pos=(468, 475)
original_word='program', pos=(516, 523)
original_word='program', pos=(533, 540)

Term: application
original_word='application', pos=(80, 91)
original_word='application', pos=(164, 175)
original_word='applications', pos=(322, 334)

Term: applet
original_word='applets', pos=(302, 309)

Term: compile
original_word='compile', pos=(56, 63)
original_word='compiling', pos=(134, 143)
original_word='Compile', pos=(504, 511)


Teraz czas zająć się problemem przygotowania słownika specjalistycznego. W tym celu napiszemy nasz własny ekstraktor terminologii. Wejściem do ekstraktora będzie tekst zawierający specjalistyczną terminologię. Wyjściem - lista terminów.

Przyjmijmy następujące podejście - terminami specjalistycznymi będą najcześćiej występujące rzeczowniki w tekście. Wykonajmy krok pierwszy:

### Ćwiczenie 4: Wypisz wszystkie rzeczowniki z tekstu. Wykorzystaj możliwości spaCy.

In [11]:
def get_nouns(text):
    doc = nlp(text)
    nouns = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]]
    return nouns

all_nouns = get_nouns(text)

print(f"Nouns count: {len(all_nouns)}")
print(all_nouns)

Nouns count: 32
['Java', 'programmers', 'section', 'Swing', 'application', 'command', 'line', 'information', 'Swing', 'application', 'NetBeans', 'IDE', 'Tutorial', 'Examples', 'NetBeans', 'IDE', 'compilation', 'instructions', 'Swing', 'programs', 'applets', 'applications', 'steps', 'release', 'Java', 'SE', 'platform', 'program', 'Swing', 'components', 'program', 'program']


Teraz czas na podliczenie wystąpień poszczególnych rzeczowników. Uwaga - różne formy tego samego słowa zliczamy razem jako wystąpienia tego słowa (np. "program" i "programs"). Najwygodniejszą metodą podliczania jest zastosowanie tzw. tally (po polsku "zestawienie"). Jest to słownik, którego kluczem jest słowo w formie podstawowej, a wartością liczba wystąpień tego słowa, wliczając słowa odmienione. Przykład gotowego tally:

In [18]:
tally = {"program" : 4, "component" : 1}

### Ćwiczenie 5: Napisz program do ekstrakcji terminologii z tekstu według powyższych wytycznych.

In [13]:
def extract_terms(text):
    doc = nlp(text)

    tally = {}
    
    for token in doc:
        if token.pos_ in ["NOUN", "PROPN"]:
            lemma = token.lemma_.lower()
        
            if lemma in tally:
                tally[lemma] += 1
            else:
                tally[lemma] = 1
                
    return tally

print(extract_terms(text))

{'java': 2, 'programmer': 1, 'section': 1, 'swing': 4, 'application': 3, 'command': 1, 'line': 1, 'information': 1, 'netbeans': 2, 'ide': 2, 'tutorial': 1, 'examples': 1, 'compilation': 1, 'instruction': 1, 'program': 4, 'applet': 1, 'step': 1, 'release': 1, 'se': 1, 'platform': 1, 'component': 1}


### Ćwiczenie 6: Rozszerz powyższy program o ekstrację czasowników i przymiotników.

In [14]:
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")

def extract_terms(text):
    doc = nlp(text)
    allowed_pos = ["NOUN", "PROPN", "VERB", "ADJ"]
    
    lemmas = [
        token.lemma_.lower() 
        for token in doc 
        if token.pos_ in allowed_pos
    ]
    
    return dict(Counter(lemmas))

print(extract_terms(text))

{'java': 2, 'programmer': 1, 'section': 1, 'explain': 1, 'compile': 3, 'run': 4, 'swing': 4, 'application': 3, 'command': 1, 'line': 1, 'information': 1, 'use': 2, 'netbeans': 2, 'ide': 2, 'see': 1, 'tutorial': 1, 'examples': 1, 'compilation': 1, 'instruction': 1, 'work': 1, 'program': 4, 'applet': 1, 'step': 1, 'need': 1, 'follow': 1, 'install': 1, 'late': 1, 'release': 1, 'se': 1, 'platform': 1, 'do': 1, 'create': 1, 'component': 1}
